In [29]:
# @title
# Instala las bibliotecas necesarias
!pip install google-api-python-client pandas --quiet

print(" [1] Instalacion de Bibliotecas lista")

 [1] Instalacion de Bibliotecas lista


In [30]:
# @title
import pandas as pd
from googleapiclient.discovery import build
from google.colab import userdata

# Obtén la clave API de tus secretos de Colab
API_KEY = userdata.get('YOUTUBE_API_KEY')

# Construye el cliente de la API de YouTube
youtube = build('youtube', 'v3', developerKey=API_KEY)

print(" [2] Cliente de la API de YouTube configurado")

 [2] Cliente de la API de YouTube configurado


In [31]:
# @title
import pandas as pd
from googleapiclient.discovery import build
from google.colab import userdata

# Define the filename for persistence
PLAYLIST_FILENAME = 'playlist_data.csv' # This will be overridden by the next cell for Drive persistence

# These functions are now defined and managed in cell 8dcb2705 for Drive persistence.
# The definitions here are effectively ignored after cell 8dcb2705 runs, but kept for context if this cell is run independently.
def save_playlist():
    global playlist_df
    try:
        playlist_df.to_csv(PLAYLIST_FILENAME, index=False)
    except Exception as e:
        print(f"Error al guardar la playlist: {e}")

def load_playlist():
    global playlist_df
    try:
        playlist_df = pd.read_csv(PLAYLIST_FILENAME)
        playlist_df['views'] = playlist_df['views'].astype('Int64')
        playlist_df['likes'] = playlist_df['likes'].astype('Int64')
        playlist_df['comment_count'] = playlist_df['comment_count'].astype('Int64')
        playlist_df['saved_views'] = playlist_df['saved_views'].astype('Int64')
        playlist_df['views_gained'] = playlist_df['views_gained'].astype('Int64')
        playlist_df['saved_likes'] = playlist_df['saved_likes'].astype('Int64')
        playlist_df['likes_gained'] = playlist_df['likes_gained'].astype('Int64')
        playlist_df['previous_rank'] = playlist_df['previous_rank'].astype('Int64')
        print(f"Playlist cargada desde {PLAYLIST_FILENAME}")
    except FileNotFoundError:
        print("No se encontró un archivo de playlist guardado. Creando una nueva playlist.")
        playlist_df = pd.DataFrame(columns=['video_id', 'url', 'title', 'views', 'likes', 'comment_count', 'composer', 'singer', 'saved_views', 'views_gained', 'saved_likes', 'likes_gained', 'previous_rank'])
    except Exception as e:
        print(f"Error al cargar la playlist, creando una nueva: {e}")
        playlist_df = pd.DataFrame(columns=['video_id', 'url', 'title', 'views', 'likes', 'comment_count', 'composer', 'singer', 'saved_views', 'views_gained', 'saved_likes', 'likes_gained', 'previous_rank'])

# --- Initialization ---
# Obtain API Key and build YouTube API client (as before)
API_KEY = userdata.get('YOUTUBE_API_KEY')
youtube = build('youtube', 'v3', developerKey=API_KEY)

# Load the playlist at startup
load_playlist()

def get_video_id_from_url(url):
    import re
    match = re.search(r'(?:v=|\&|[\?]|/)([a-zA-Z0-9_-]{11})(?:&.*)?$', url)
    if match:
        return match.group(1)
    return None

def get_video_details(video_id):
    try:
        request = youtube.videos().list(
            part='snippet,statistics',
            id=video_id
        )
        response = request.execute()

        if response and response['items']:
            item = response['items'][0]
            title = item['snippet']['title']
            views = int(item['statistics'].get('viewCount', 0))
            likes = int(item['statistics'].get('likeCount', 0))
            comment_count = int(item['statistics'].get('commentCount', 0))
            return title, views, likes, comment_count
        return None, None, None, None
    except Exception as e:
        print(f"Error al obtener detalles para el video {video_id}: {e}")
        return None, None, None, None

print(" [3] Cliente de la API de YouTube configurado y DataFrame de playlist inicializado/cargado.")

No se encontró un archivo de playlist guardado. Creando una nueva playlist.
 [3] Cliente de la API de YouTube configurado y DataFrame de playlist inicializado/cargado.


In [32]:
  # @title
def add_video_to_playlist(url, titulo_personalizado=None, composer=None, singer=None):
    """Añade un nuevo video a la playlist o actualiza si ya existe, con un título personalizado opcional, compositor y cantante."""
    global playlist_df
    video_id = get_video_id_from_url(url)

    if not video_id:
        print(f"URL inválida: {url}")
        return

    # Check if video already exists
    if video_id in playlist_df['video_id'].values:
        idx = playlist_df.index[playlist_df['video_id'] == video_id].tolist()[0]
        api_title, new_views, new_likes, new_comment_count, new_published_at = get_video_details(video_id)

        if api_title is not None:
            if titulo_personalizado is not None:
                playlist_df.at[idx, 'title'] = titulo_personalizado
            elif playlist_df.at[idx, 'title'] == api_title or pd.isna(playlist_df.at[idx, 'title']):
                playlist_df.at[idx, 'title'] = api_title

            # Update current metrics
            playlist_df.at[idx, 'views'] = new_views
            playlist_df.at[idx, 'likes'] = new_likes
            playlist_df.at[idx, 'comment_count'] = new_comment_count
            playlist_df.at[idx, 'published_at'] = pd.to_datetime(new_published_at)

            # Si no tiene métricas guardadas, por defecto usamos los valores actuales de la API
            if pd.isna(playlist_df.at[idx, 'saved_views']):
                 playlist_df.at[idx, 'saved_views'] = new_views
            if pd.isna(playlist_df.at[idx, 'saved_likes']):
                 playlist_df.at[idx, 'saved_likes'] = new_likes
            if pd.isna(playlist_df.at[idx, 'saved_comment_count']):
                 playlist_df.at[idx, 'saved_comment_count'] = new_comment_count

            playlist_df.at[idx, 'views_gained'] = new_views - playlist_df.at[idx, 'saved_views']
            playlist_df.at[idx, 'likes_gained'] = new_likes - playlist_df.at[idx, 'saved_likes']
            playlist_df.at[idx, 'comment_gained'] = new_comment_count - playlist_df.at[idx, 'saved_comment_count']

            if composer is not None: playlist_df.at[idx, 'composer'] = composer
            if singer is not None: playlist_df.at[idx, 'singer'] = singer

            print(f"Video '{playlist_df.at[idx, 'title']}' actualizado.")
        else:
            print(f"No se pudo obtener información actualizada para el video con ID {video_id}.")
        return

    # If video is new
    api_title, views, likes, comment_count, published_at = get_video_details(video_id)
    title_to_use = titulo_personalizado if titulo_personalizado is not None else api_title

    views_val = pd.NA if views is None else views
    likes_val = pd.NA if likes is None else likes
    comment_count_val = pd.NA if comment_count is None else comment_count

    if title_to_use is not None:
        import pytz
        new_york_timezone = pytz.timezone('America/New_York')
        now_ny = pd.Timestamp.now(tz=new_york_timezone)
        current_month_start_ny = now_ny.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
        end_of_previous_month_ny = current_month_start_ny - pd.Timedelta(microseconds=1)
        start_of_previous_seven_days_ny = current_month_start_ny - pd.Timedelta(days=7)

        published_at_dt = pd.to_datetime(published_at)
        published_at_ny = published_at_dt.tz_localize(new_york_timezone) if published_at_dt.tz is None else published_at_dt.tz_convert(new_york_timezone)

        # Un video se considera candidato a [NEW] por fecha de estreno
        is_new_video_by_date = (published_at_ny >= current_month_start_ny) or \
                               (published_at_ny >= start_of_previous_seven_days_ny and published_at_ny <= end_of_previous_month_ny)

        # Si es un estreno real, su punto de partida guardado es 0 para que views_gained sea igual a sus views
        # Si es una canción antigua que recién agregamos, se inicializa con sus vistas actuales como base (vistas ganadas inician en 0)
        if is_new_video_by_date:
            saved_views_init = 0
            saved_likes_init = 0
            saved_comments_init = 0
            views_gained_init = views_val
            likes_gained_init = likes_val
            comments_gained_init = comment_count_val
        else:
            saved_views_init = views_val
            saved_likes_init = likes_val
            saved_comments_init = comment_count_val
            views_gained_init = 0
            likes_gained_init = 0
            comments_gained_init = 0

        new_row = pd.DataFrame(
            [
                {
                    'video_id': video_id,
                    'url': url,
                    'title': title_to_use,
                    'views': views_val,
                    'likes': likes_val,
                    'comment_count': comment_count_val,
                    'composer': composer,
                    'singer': singer,
                    'published_at': pd.to_datetime(published_at),
                    'saved_views': saved_views_init,
                    'views_gained': views_gained_init,
                    'saved_likes': saved_likes_init,
                    'likes_gained': likes_gained_init,
                    'saved_comment_count': saved_comments_init,
                    'comment_gained': comments_gained_init,
                    'last_reset_date': pd.NaT,
                    'saved_rank_gained': pd.NA
                }
            ]
        )
        playlist_df = pd.concat([playlist_df, new_row], ignore_index=True)
        print(f"Video '{title_to_use}' añadido.")
    else:
         print(f"No se pudo obtener información para el video con ID {video_id}.")

def update_metrics():
    """Actualiza el recuento de vistas, likes, comentarios y calcula las vistas y likes ganados para todos los videos en la playlist."""
    global playlist_df
    if playlist_df.empty:
        print("La playlist está vacía. No hay métricas para actualizar.")
        return

    print("Actualizando vistas, likes, comentarios y métricas ganadas...")
    import pytz
    new_york_timezone = pytz.timezone('America/New_York')
    now_ny = pd.Timestamp.now(tz=new_york_timezone)
    current_month_start_ny = now_ny.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
    end_of_previous_month_ny = current_month_start_ny - pd.Timedelta(microseconds=1)
    start_of_previous_seven_days_ny = current_month_start_ny - pd.Timedelta(days=7)

    is_reset_day = now_ny.day == 1

    rank_map_gained_for_reset = {}
    if is_reset_day:
        temp_df_for_rank = playlist_df.copy().dropna(subset=['views_gained'])
        if not temp_df_for_rank.empty:
            temp_df_for_rank = temp_df_for_rank.sort_values(by='views_gained', ascending=False).reset_index(drop=True)
            temp_df_for_rank['current_rank_gained'] = temp_df_for_rank.index + 1
            rank_map_gained_for_reset = temp_df_for_rank.set_index('video_id')['current_rank_gained'].to_dict()

    for index, row in playlist_df.iterrows():
        video_id = row['video_id']
        api_title, raw_api_views, raw_api_likes, raw_api_comment_count, published_at_str = get_video_details(video_id)

        if api_title is not None:
            published_at = pd.to_datetime(published_at_str)
            published_at_ny = published_at.tz_localize(new_york_timezone) if published_at.tz is None else published_at.tz_convert(new_york_timezone)

            # Un video se considera candidato a [NEW] por fecha de estreno
            is_new_video_by_date = (published_at_ny >= current_month_start_ny) or \
                                   (published_at_ny >= start_of_previous_seven_days_ny and published_at_ny <= end_of_previous_month_ny)

            final_views = raw_api_views
            final_likes = raw_api_likes
            final_comment_count = raw_api_comment_count
            final_views_gained = 0
            final_likes_gained = 0
            final_comment_gained = 0

            # Caso 1: Es el día de reinicio mensual y aún no ha sido procesado este mes
            if is_reset_day and (pd.isna(row['last_reset_date']) or row['last_reset_date'].month != now_ny.month or row['last_reset_date'].year != now_ny.year):
                print(f"Reiniciando métricas para el video '{row['title']}' (ID: {video_id}) para el nuevo mes.")

                # Al pasar por el reinicio, se guarda su estado actual como base para comenzar desde 0 el nuevo mes
                playlist_df.at[index, 'saved_views'] = raw_api_views
                playlist_df.at[index, 'saved_likes'] = raw_api_likes
                playlist_df.at[index, 'saved_comment_count'] = raw_api_comment_count

                playlist_df.at[index, 'views_gained'] = 0
                playlist_df.at[index, 'likes_gained'] = 0
                playlist_df.at[index, 'comment_gained'] = 0
                playlist_df.at[index, 'last_reset_date'] = now_ny.normalize()
                playlist_df.at[index, 'saved_rank_gained'] = rank_map_gained_for_reset.get(video_id, pd.NA)

                final_views_gained = 0
                final_likes_gained = 0
                final_comment_gained = 0

            # Caso 2: Es un video de estreno que todavía no ha pasado por el reinicio mensual en curso
            elif is_new_video_by_date and (pd.isna(row['last_reset_date']) or row['last_reset_date'].month != now_ny.month or row['last_reset_date'].year != now_ny.year):
                # Conserva sus vistas acumuladas completas
                playlist_df.at[index, 'saved_views'] = 0
                playlist_df.at[index, 'saved_likes'] = 0
                playlist_df.at[index, 'saved_comment_count'] = 0

                final_views_gained = raw_api_views
                final_likes_gained = raw_api_likes
                final_comment_gained = raw_api_comment_count

            # Caso 3: Actualización rutinaria de cualquier otro video (antiguo o ya reiniciado)
            else:
                # Si un video antiguo tenía guardado saved_views en 0 por error, corregimos su base con sus vistas actuales
                # para que no muestre una ganancia artificial gigante y sus vistas ganadas comiencen en 0.
                if not is_new_video_by_date and (row['saved_views'] == 0 or pd.isna(row['saved_views'])):
                    playlist_df.at[index, 'saved_views'] = raw_api_views
                    playlist_df.at[index, 'saved_likes'] = raw_api_likes
                    playlist_df.at[index, 'saved_comment_count'] = raw_api_comment_count

                    current_saved_views_for_gain_calc = raw_api_views
                    current_saved_likes_for_gain_calc = raw_api_likes
                    current_saved_comment_count_for_gain_calc = raw_api_comment_count
                else:
                    current_saved_views_for_gain_calc = row['saved_views'] if pd.notna(row['saved_views']) else raw_api_views
                    current_saved_likes_for_gain_calc = row['saved_likes'] if pd.notna(row['saved_likes']) else raw_api_likes
                    current_saved_comment_count_for_gain_calc = row['saved_comment_count'] if pd.notna(row['saved_comment_count']) else raw_api_comment_count

                    if pd.isna(row['saved_views']): playlist_df.at[index, 'saved_views'] = raw_api_views
                    if pd.isna(row['saved_likes']): playlist_df.at[index, 'saved_likes'] = raw_api_likes
                    if pd.isna(row['saved_comment_count']): playlist_df.at[index, 'saved_comment_count'] = raw_api_comment_count

                final_views_gained = raw_api_views - current_saved_views_for_gain_calc
                final_likes_gained = raw_api_likes - current_saved_likes_for_gain_calc
                final_comment_gained = raw_api_comment_count - current_saved_comment_count_for_gain_calc

            if (row['title'] == api_title) or (pd.isna(row['title']) and api_title is not None):
                 playlist_df.at[index, 'title'] = api_title

            playlist_df.at[index, 'views'] = final_views
            playlist_df.at[index, 'likes'] = final_likes
            playlist_df.at[index, 'comment_count'] = final_comment_count
            playlist_df.at[index, 'published_at'] = published_at

            playlist_df.at[index, 'views_gained'] = final_views_gained
            playlist_df.at[index, 'likes_gained'] = final_likes_gained
            playlist_df.at[index, 'comment_gained'] = final_comment_gained

        else:
            print(f"Advertencia: No se pudo obtener información actualizada para el video {video_id}.")
    print("Métricas actualizadas.")
    save_playlist()

def display_playlist():
    """Muestra la playlist ordenada por vistas de forma descendente."""
    global playlist_df
    if playlist_df.empty:
        print("La playlist está vacía.")
        return

    filtered_df = playlist_df[playlist_df['views'].notna()].copy()
    if filtered_df.empty:
        print("La playlist está vacía después de filtrar videos sin vistas para mostrar.")
        return

    temp_df = filtered_df.copy()

    temp_df_for_initial_rank = temp_df[temp_df['saved_views'].notna()].copy()
    if not temp_df_for_initial_rank.empty:
        temp_df_for_initial_rank = temp_df_for_initial_rank.sort_values(by='saved_views', ascending=False).reset_index(drop=True)
        temp_df_for_initial_rank['initial_rank'] = temp_df_for_initial_rank.index + 1
        initial_rank_map = temp_df_for_initial_rank.set_index('video_id')['initial_rank']
        temp_df['initial_rank_views'] = temp_df['video_id'].map(initial_rank_map)
    else:
        temp_df['initial_rank_views'] = pd.NA

    temp_df = temp_df.sort_values(by='views', ascending=False).reset_index(drop=True)
    temp_df['Posición'] = temp_df.index + 1
    temp_df['Cambio'] = 'N/A'

    new_york_timezone = pytz.timezone('America/New_York')
    now_ny = pd.Timestamp.now(tz=new_york_timezone)
    current_month_start_ny = now_ny.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
    end_of_previous_month_ny = current_month_start_ny - pd.Timedelta(microseconds=1)
    start_of_previous_seven_days_ny = current_month_start_ny - pd.Timedelta(days=7)

    for idx, row in temp_df.iterrows():
        current_rank = row['Posición']
        initial_rank = row['initial_rank_views']
        last_reset = row['last_reset_date']

        # Condición corregida: Solo es [NEW] si se estrenó en el rango Y NO ha sido alcanzado por el reinicio del mes actual
        has_been_reset_this_month = pd.notna(last_reset) and (last_reset.month == now_ny.month) and (last_reset.year == now_ny.year)

        is_new_by_date = False
        if pd.notna(row['published_at']):
            published_at = pd.to_datetime(row['published_at'])
            published_at_ny = published_at.tz_localize(new_york_timezone) if published_at.tz is None else published_at.tz_convert(new_york_timezone)
            if (published_at_ny >= current_month_start_ny) or \
               (published_at_ny >= start_of_previous_seven_days_ny and published_at_ny <= end_of_previous_month_ny):
                is_new_by_date = True

        if is_new_by_date and not has_been_reset_this_month:
            temp_df.at[idx, 'Cambio'] = "[NEW]"
            continue

        if pd.notna(initial_rank):
            change = int(initial_rank - current_rank)
            if change > 0:
                temp_df.at[idx, 'Cambio'] = f"↑{change}"
            elif change < 0:
                temp_df.at[idx, 'Cambio'] = f"↓{abs(change)}"
            else:
                temp_df.at[idx, 'Cambio'] = "—"
        else:
            temp_df.at[idx, 'Cambio'] = "—"

    def format_number_with_dots(num):
        if pd.isna(num):
            return "N/A"
        return f"{int(num):,}".replace(",", ".")

    def format_date(date_val):
        if pd.isna(date_val) or pd.isnull(date_val):
            return "N/A"
        return pd.to_datetime(date_val).strftime('%m-%d-%Y')

    display_df = temp_df.copy()
    display_df['Vistas Actuales'] = display_df['views'].apply(format_number_with_dots)
    display_df['Me Gusta'] = display_df['likes'].apply(format_number_with_dots)
    display_df['Comentarios'] = display_df['comment_count'].apply(format_number_with_dots)
    display_df['Vistas Ganadas'] = display_df['views_gained'].apply(format_number_with_dots)
    display_df['Me Gusta Ganados'] = display_df['likes_gained'].apply(format_number_with_dots)
    display_df['Comentarios Ganados'] = display_df['comment_gained'].apply(format_number_with_dots)
    display_df['Fecha de Publicación'] = display_df['published_at'].apply(format_date)
    display_df = display_df.rename(columns={
        'title': 'Título',
        'composer': 'Compositor',
        'singer': 'Cantante'
    })

    display_df = display_df.set_index('Posición')

    print("\n--- Tu Playlist (ordenada por vistas) ---")
    display(display_df[['Cambio', 'Título', 'Compositor', 'Cantante', 'Fecha de Publicación', 'Vistas Actuales', 'Vistas Ganadas', 'Me Gusta', 'Me Gusta Ganados', 'Comentarios', 'Comentarios Ganados']])

print(" [4] Definición de funciones de gestión de la playlist lista (Lógica [NEW] corregida y mejorada).")

 [4] Definición de funciones de gestión de la playlist lista (Lógica [NEW] corregida y mejorada).


In [33]:
  # @title
def display_playlist_by_views_gained():
    """Muestra la playlist ordenada por vistas ganadas de forma descendente, filtrando videos sin vistas y mostrando cambio de posición."""
    global playlist_df
    if playlist_df.empty:
        print("La playlist está vacía. No hay vistas ganadas para mostrar.")
        return

    filtered_df = playlist_df[playlist_df['views'].notna()].copy()
    if filtered_df.empty:
        print("La playlist está vacía después de filtrar videos sin vistas para mostrar.")
        return

    temp_df = filtered_df.copy()
    temp_df = temp_df.sort_values(by='views_gained', ascending=False).reset_index(drop=True)
    temp_df['Posición'] = temp_df.index + 1
    temp_df['Cambio'] = 'N/A'

    new_york_timezone = pytz.timezone('America/New_York')
    now_ny = pd.Timestamp.now(tz=new_york_timezone)
    current_month_start_ny = now_ny.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
    end_of_previous_month_ny = current_month_start_ny - pd.Timedelta(microseconds=1)
    start_of_previous_seven_days_ny = current_month_start_ny - pd.Timedelta(days=7)

    for idx, row in temp_df.iterrows():
        current_rank = row['Posición']
        previous_rank_gained = row['saved_rank_gained']
        last_reset = row['last_reset_date']

        # Condición corregida: Solo conserva [NEW] si NO ha sido afectado por el reinicio del mes en curso
        has_been_reset_this_month = pd.notna(last_reset) and (last_reset.month == now_ny.month) and (last_reset.year == now_ny.year)

        is_new_by_date = False
        if pd.notna(row['published_at']):
            published_at = pd.to_datetime(row['published_at'])
            published_at_ny = published_at.tz_localize(new_york_timezone) if published_at.tz is None else published_at.tz_convert(new_york_timezone)
            if (published_at_ny >= current_month_start_ny) or \
               (published_at_ny >= start_of_previous_seven_days_ny and published_at_ny <= end_of_previous_month_ny):
                is_new_by_date = True

        if is_new_by_date and not has_been_reset_this_month:
            temp_df.at[idx, 'Cambio'] = "[NEW]"
            continue

        if pd.notna(previous_rank_gained):
            change = int(previous_rank_gained) - int(current_rank)
            if change > 0:
                temp_df.at[idx, 'Cambio'] = f"↑{change}"
            elif change < 0:
                temp_df.at[idx, 'Cambio'] = f"↓{abs(change)}"
            else:
                temp_df.at[idx, 'Cambio'] = "—"
        else:
            temp_df.at[idx, 'Cambio'] = "—"

    def format_number_with_dots(num):
        if pd.isna(num):
            return "N/A"
        return f" {int(num):,}".replace(",", ".")

    def format_date(date_val):
        if pd.isna(date_val) or pd.isnull(date_val):
            return "N/A"
        return pd.to_datetime(date_val).strftime('%m-%d-%Y')

    def format_previous_rank(rank_val, has_been_reset_this_month, is_new_by_date):
        if is_new_by_date and not has_been_reset_this_month:
            return "[NEW]"
        if pd.isna(rank_val):
            return "—"
        return str(int(rank_val))

    display_df = temp_df.copy()
    display_df['Vistas Actuales'] = display_df['views'].apply(format_number_with_dots)
    display_df['Me Gusta'] = display_df['likes'].apply(format_number_with_dots)
    display_df['Comentarios'] = display_df['comment_count'].apply(format_number_with_dots)
    display_df['Vistas Ganadas'] = display_df['views_gained'].apply(format_number_with_dots)
    display_df['Me Gusta Ganados'] = display_df['likes_gained'].apply(format_number_with_dots)
    display_df['Comentarios Ganados'] = display_df['comment_gained'].apply(format_number_with_dots)
    display_df['Fecha de Publicación'] = display_df['published_at'].apply(format_date)

    # Aplicar la lógica dinámica para el formato de posición anterior
    pos_anterior_list = []
    for idx, row in temp_df.iterrows():
        last_reset = row['last_reset_date']
        has_been_reset_this_month = pd.notna(last_reset) and (last_reset.month == now_ny.month) and (last_reset.year == now_ny.year)
        is_new_by_date = False
        if pd.notna(row['published_at']):
            published_at = pd.to_datetime(row['published_at'])
            published_at_ny = published_at.tz_localize(new_york_timezone) if published_at.tz is None else published_at.tz_convert(new_york_timezone)
            if (published_at_ny >= current_month_start_ny) or \
               (published_at_ny >= start_of_previous_seven_days_ny and published_at_ny <= end_of_previous_month_ny):
                is_new_by_date = True

        pos_anterior_list.append(format_previous_rank(row['saved_rank_gained'], has_been_reset_this_month, is_new_by_date))

    display_df['Posición Anterior'] = pos_anterior_list

    display_df = display_df.rename(columns={
        'title': 'Título',
        'composer': 'Compositor',
        'singer': 'Cantante'
    })

    display_df = display_df.set_index('Posición')

    print("\n--- Tu Playlist (ordenada por Vistas Ganadas - Ranking Mensualizado) ---")
    display(display_df[['Cambio', 'Título', 'Compositor', 'Cantante', 'Fecha de Publicación', 'Vistas Actuales', 'Vistas Ganadas', 'Me Gusta', 'Comentarios', 'Posición Anterior']])

    display_time_until_reset()

    top_100_df = temp_df.head(100).copy()

    # --- ALGORITMO DE ESTANDARIZACIÓN INTELIGENTE DE NOMBRES POR FRECUENCIA (Top 100) ---
    def get_standardization_mapping(series_col):
        """Crea un diccionario que mapea variantes de nombres (ignorando mayúsculas/minúsculas)
        a la variante exacta más común en los datos."""
        if series_col.dropna().empty:
            return {}

        # Desglosar los nombres por si vienen separados por comas
        all_names = series_col.astype(str).str.split(',').explode().str.strip().replace('', pd.NA).dropna()
        if all_names.empty:
            return {}

        # Contar cuántas veces aparece cada representación exacta
        counts = all_names.value_counts()

        # Agrupar las representaciones exactas por su versión en minúsculas
        grouped_by_lower = {}
        for name, count in counts.items():
            name_lower = name.lower()
            if name_lower not in grouped_by_lower:
                grouped_by_lower[name_lower] = []
            grouped_by_lower[name_lower].append((name, count))

        # Crear un mapa: variante en minúsculas -> variante exacta más repetida
        standard_map = {}
        for name_lower, variants in grouped_by_lower.items():
            # Ordenar variantes por frecuencia (descendente) y elegir la de mayor frecuencia
            best_variant = sorted(variants, key=lambda x: x[1], reverse=True)[0][0]
            standard_map[name_lower] = best_variant

        return standard_map

    # Obtener mapas de estandarización sobre los datos del Top 100
    composer_std_map = get_standardization_mapping(top_100_df['composer'])
    singer_std_map = get_standardization_mapping(top_100_df['singer'])

    def standardize_list(cell_val, std_map):
        if pd.isna(cell_val):
            return cell_val
        parts = [p.strip() for p in str(cell_val).split(',') if p.strip()]
        standardized_parts = []
        for p in parts:
            p_lower = p.lower()
            if p_lower in std_map:
                standardized_parts.append(std_map[p_lower])
            else:
                standardized_parts.append(p)
        return ", ".join(standardized_parts)

    # Aplicar estandarización para el reporte
    if 'composer' in top_100_df.columns:
        top_100_df['composer_std'] = top_100_df['composer'].apply(lambda x: standardize_list(x, composer_std_map))
    else:
        top_100_df['composer_std'] = pd.NA

    if 'singer' in top_100_df.columns:
        top_100_df['singer_std'] = top_100_df['singer'].apply(lambda x: standardize_list(x, singer_std_map))
    else:
        top_100_df['singer_std'] = pd.NA

    print("\n--- Resumen de Compositores (Top 100 por Vistas Ganadas - Estandarizado) ---")
    if not top_100_df.empty:
        if 'composer_std' in top_100_df.columns and not top_100_df['composer_std'].dropna().empty:
            processed_composers = top_100_df['composer_std'].astype(str).str.split(',').explode().str.strip().replace('', pd.NA).dropna()
            composer_counts = processed_composers.value_counts()
            repeated_composers = composer_counts[composer_counts >= 3]
            if not repeated_composers.empty:
                print("Compositores con 3 o más apariciones en el Top 100 (unificados por tipografía más común):")
                for composer, count in repeated_composers.items():
                    print(f"  - {composer}: {count} Canciones")
            else:
                print("Ningún compositor tiene 3 o más canciones en el Top 100.")
        else:
            print("No hay datos de compositores en el Top 100.")
    else:
        print("No hay suficientes datos para generar el resumen de compositores.")

    print("\n--- Resumen de Cantantes (Top 100 por Vistas Ganadas - Estandarizado) ---")
    if not top_100_df.empty:
        if 'singer_std' in top_100_df.columns and not top_100_df['singer_std'].dropna().empty:
            processed_singers = top_100_df['singer_std'].astype(str).str.split(',').explode().str.strip().replace('', pd.NA).dropna()
            singer_counts = processed_singers.value_counts()
            repeated_singers = singer_counts[singer_counts >= 3]
            if not repeated_singers.empty:
                print("Cantantes con 3 o más apariciones en el Top 100 (unificados por tipografía más común):")
                for singer, count in repeated_singers.items():
                    print(f"  - {singer}: {count} Canciones")
            else:
                print("Ningún cantante tiene 3 o más canciones en el Top 100.")
        else:
            print("No hay datos de cantantes en el Top 100.")
    else:
        print("No hay suficientes datos para generar el resumen de cantantes.")

print(" [5] Definición de funciones de visualización de rankings lista (Estandarización inteligente de artistas añadida por frecuencia).")

 [5] Definición de funciones de visualización de rankings lista (Estandarización inteligente de artistas añadida por frecuencia).


In [34]:
# @title
from IPython.display import display, HTML
import pandas as pd
import pytz

try:
    from google.colab import data_table
except ImportError:
    pass

def display_playlist_by_views_gained():
    """Muestra la playlist completa ordenada por vistas ganadas de forma descendente, con formato de Crecimiento Combinado y formato interactivo de Colab."""
    global playlist_df
    if playlist_df.empty:
        print("La playlist está vacía. No hay vistas ganadas para mostrar.")
        return

    filtered_df = playlist_df[playlist_df['views'].notna()].copy()
    if filtered_df.empty:
        print("La playlist está vacía después de filtrar videos sin vistas para mostrar.")
        return

    temp_df = filtered_df.copy()
    temp_df = temp_df.sort_values(by='views_gained', ascending=False).reset_index(drop=True)
    temp_df['Posición'] = temp_df.index + 1
    temp_df['Cambio'] = 'N/A'

    new_york_timezone = pytz.timezone('America/New_York')
    now_ny = pd.Timestamp.now(tz=new_york_timezone)
    current_month_start_ny = now_ny.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
    end_of_previous_month_ny = current_month_start_ny - pd.Timedelta(microseconds=1)
    start_of_previous_seven_days_ny = current_month_start_ny - pd.Timedelta(days=7)

    for idx, row in temp_df.iterrows():
        current_rank = row['Posición']
        previous_rank_gained = row['saved_rank_gained']
        last_reset = row['last_reset_date']

        has_been_reset_this_month = pd.notna(last_reset) and (last_reset.month == now_ny.month) and (last_reset.year == now_ny.year)

        is_new_by_date = False
        if pd.notna(row['published_at']):
            published_at = pd.to_datetime(row['published_at'])
            published_at_ny = published_at.tz_localize(new_york_timezone) if published_at.tz is None else published_at.tz_convert(new_york_timezone)
            if (published_at_ny >= current_month_start_ny) or \
               (published_at_ny >= start_of_previous_seven_days_ny and published_at_ny <= end_of_previous_month_ny):
                is_new_by_date = True

        if is_new_by_date and not has_been_reset_this_month:
            temp_df.at[idx, 'Cambio'] = "[NEW]"
            continue

        if pd.notna(previous_rank_gained):
            change = int(previous_rank_gained) - int(current_rank)
            if change > 0:
                temp_df.at[idx, 'Cambio'] = f"↑{change}"
            elif change < 0:
                temp_df.at[idx, 'Cambio'] = f"↓{abs(change)}"
            else:
                temp_df.at[idx, 'Cambio'] = "—"
        else:
            temp_df.at[idx, 'Cambio'] = "—"

    def format_combined_views(row_df):
        gained = row_df['views_gained']
        actuals = row_df['views']
        gained_str = f"{int(gained):,}".replace(",", ".") if pd.notna(gained) else "0"
        actuals_str = f"{int(actuals):,}".replace(",", ".") if pd.notna(actuals) else "0"
        return f"+{gained_str}   ({actuals_str})"

    def format_number_with_dots(num):
        if pd.isna(num):
            return "N/A"
        return f"{int(num):,}".replace(",", ".")

    def format_date(date_val):
        if pd.isna(date_val) or pd.isnull(date_val):
            return "N/A"
        return pd.to_datetime(date_val).strftime('%m-%d-%Y')

    def format_previous_rank(rank_val, has_been_reset_this_month, is_new_by_date):
        if is_new_by_date and not has_been_reset_this_month:
            return "[NEW]"
        if pd.isna(rank_val):
            return "—"
        return str(int(rank_val))

    display_df = temp_df.copy()
    display_df['Vistas (Crecimiento)'] = display_df.apply(format_combined_views, axis=1)
    display_df['Me Gusta'] = display_df['likes'].apply(format_number_with_dots)
    display_df['Comentarios'] = display_df['comment_count'].apply(format_number_with_dots)
    display_df['Fecha de Publicación'] = display_df['published_at'].apply(format_date)

    pos_anterior_list = []
    for idx, row in temp_df.iterrows():
        last_reset = row['last_reset_date']
        has_been_reset_this_month = pd.notna(last_reset) and (last_reset.month == now_ny.month) and (last_reset.year == now_ny.year)
        is_new_by_date = False
        if pd.notna(row['published_at']):
            published_at = pd.to_datetime(row['published_at'])
            published_at_ny = published_at.tz_localize(new_york_timezone) if published_at.tz is None else published_at.tz_convert(new_york_timezone)
            if (published_at_ny >= current_month_start_ny) or \
               (published_at_ny >= start_of_previous_seven_days_ny and published_at_ny <= end_of_previous_month_ny):
                is_new_by_date = True

        pos_anterior_list.append(format_previous_rank(row['saved_rank_gained'], has_been_reset_this_month, is_new_by_date))

    display_df['Posición Anterior'] = pos_anterior_list

    display_df = display_df.rename(columns={
        'title': 'Título',
        'composer': 'Compositor',
        'singer': 'Cantante'
    })

    display_df = display_df.set_index('Posición')

    print("\n--- Tu Playlist Completa (Vistas Ganadas - Ranking Mensualizado Interactivo) ---")
    ordered_columns = ['Cambio', 'Título', 'Compositor', 'Cantante', 'Fecha de Publicación', 'Vistas (Crecimiento)', 'Me Gusta', 'Comentarios', 'Posición Anterior']

    # Usamos la extensión de Colab data_table para mantener el control interactivo y paginado
    try:
        interactive_table = data_table.DataTable(display_df[ordered_columns], num_rows_per_page=15, max_columns=10)
        display(interactive_table)
    except:
        display(display_df[ordered_columns])

print(" [5] Función de visualización general simplificada (Formato de vistas combinadas aplicado con soporte interactivo para Google Colab, sin miniaturas).")

 [5] Función de visualización general simplificada (Formato de vistas combinadas aplicado con soporte interactivo para Google Colab, sin miniaturas).


In [35]:
# @title
from datetime import datetime, timedelta
import pytz

def display_time_until_reset():
    """Calcula y muestra el tiempo restante hasta el próximo reinicio mensual (día 1) en la zona horaria de Nueva York."""
    new_york_timezone = pytz.timezone('America/New_York')
    now_ny = datetime.now(new_york_timezone)

    next_reset_month = now_ny.month
    next_reset_year = now_ny.year

    # If today is the 1st or later, the next reset is in the next month
    # We are considering the 'day' of the New York timezone
    if now_ny.day >= 1:
        next_reset_month += 1
        if next_reset_month > 12:
            next_reset_month = 1
            next_reset_year += 1

    # The reset is always on the 1st of the month, at midnight NY time
    next_reset_date_ny = new_york_timezone.localize(datetime(next_reset_year, next_reset_month, 1, 0, 0, 0))

    time_remaining = next_reset_date_ny - now_ny

    days = time_remaining.days
    seconds = time_remaining.seconds
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    seconds = seconds % 60

    print(f"\n--- Tiempo restante hasta el próximo reinicio (Día 1 del mes // UTC-5): {days} días, {hours} horas, {minutes} minutos, {seconds} segundos ---")

print(" [6] Definición de funciones de visualización de tiempo lista.")

 [6] Definición de funciones de visualización de tiempo lista.


In [36]:
# @title
#!rm -rf /content/drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print(" [7] Montando Google Drive para persistencia de datos lista.")

Mounted at /content/drive
 [7] Montando Google Drive para persistencia de datos lista.


In [37]:
# @title
import pandas as pd
from googleapiclient.discovery import build
from google.colab import userdata
import os
import pytz

# Define la ruta para persistencia en Google Drive
PLAYLIST_FILENAME = '/content/drive/MyDrive/Mi_Playlist_Vocaloid/playlist_data.csv'

def save_playlist():
    """Guarda el DataFrame de la playlist en un archivo CSV."""
    global playlist_df
    try:
        os.makedirs(os.path.dirname(PLAYLIST_FILENAME), exist_ok=True)
        playlist_df.to_csv(PLAYLIST_FILENAME, index=False)
        print(f"Playlist guardada en {PLAYLIST_FILENAME}")
    except Exception as e:
        print(f"Error al guardar la playlist: {e}")

def load_playlist():
    """Carga el DataFrame de la playlist desde un archivo CSV o crea uno nuevo si no existe."""
    global playlist_df
    required_int_columns = [
        'views', 'likes', 'comment_count', 'saved_views', 'views_gained',
        'saved_likes', 'likes_gained', 'saved_comment_count', 'comment_gained',
        'saved_rank_gained'
    ]
    all_columns = ['video_id', 'url', 'title', 'views', 'likes', 'comment_count',
                   'composer', 'singer', 'saved_views', 'views_gained',
                   'saved_likes', 'likes_gained', 'saved_comment_count', 'comment_gained',
                   'last_reset_date', 'published_at',
                   'saved_rank_gained']

    try:
        playlist_df = pd.read_csv(PLAYLIST_FILENAME)

        # Asegurar que todas las columnas esperadas existan
        for col in all_columns:
            if col not in playlist_df.columns:
                playlist_df[col] = pd.NA

        # Asegurar tipos de datos correctos para numéricos
        for col in required_int_columns:
            playlist_df[col] = pd.to_numeric(playlist_df[col], errors='coerce').astype('Int64')

        # Convertir fechas
        playlist_df['last_reset_date'] = pd.to_datetime(playlist_df['last_reset_date'], errors='coerce')
        playlist_df['published_at'] = pd.to_datetime(playlist_df['published_at'], errors='coerce')

        # Inicializar saved_rank_gained si es NA
        if 'saved_rank_gained' in playlist_df.columns:
            na_rank_video_ids = playlist_df[playlist_df['saved_rank_gained'].isnull()]['video_id'].tolist()
            if na_rank_video_ids:
                print(f"DEBUG: Inicializando saved_rank_gained con rankings actuales para {len(na_rank_video_ids)} videos.")
                temp_df_for_current_rank = playlist_df.copy().dropna(subset=['views_gained'])
                if not temp_df_for_current_rank.empty:
                    temp_df_for_current_rank = temp_df_for_current_rank.sort_values(by='views_gained', ascending=False).reset_index(drop=True)
                    temp_df_for_current_rank['current_rank_gained'] = temp_df_for_current_rank.index + 1
                    current_rank_map = temp_df_for_current_rank.set_index('video_id')['current_rank_gained'].to_dict()

                    for video_id in na_rank_video_ids:
                        if video_id in current_rank_map:
                            idx = playlist_df.index[playlist_df['video_id'] == video_id].tolist()[0]
                            playlist_df.at[idx, 'saved_rank_gained'] = current_rank_map[video_id]

        print(f"Playlist cargada desde {PLAYLIST_FILENAME}")
    except FileNotFoundError:
        print("No se encontró un archivo de playlist guardado. Creando una nueva playlist.")
        playlist_df = pd.DataFrame(columns=all_columns)
        for col in required_int_columns:
            playlist_df[col] = playlist_df[col].astype('Int64')
        playlist_df['last_reset_date'] = pd.NaT
        playlist_df['published_at'] = pd.NaT
    except Exception as e:
        print(f"Error al cargar la playlist: {e}")
        playlist_df = pd.DataFrame(columns=all_columns)
        for col in required_int_columns:
            playlist_df[col] = playlist_df[col].astype('Int64')
        playlist_df['last_reset_date'] = pd.NaT
        playlist_df['published_at'] = pd.NaT

# --- Inicialización del Cliente API ---
API_KEY = userdata.get('YOUTUBE_API_KEY')
youtube = build('youtube', 'v3', developerKey=API_KEY)

# Cargar la playlist
load_playlist()

def get_video_id_from_url(url):
    """Extrae el ID de video de una URL de YouTube."""
    import re
    match = re.search(r'(?:v=|&|[?]|/)([a-zA-Z0-9_-]{11})(?:&.*)?$', url)
    if match:
        return match.group(1)
    return None

def get_video_details(video_id):
    """Obtiene detalles de un video de forma individual."""
    try:
        request = youtube.videos().list(part='snippet,statistics', id=video_id)
        response = request.execute()
        if response and response['items']:
            item = response['items'][0]
            title = item['snippet']['title']
            views = int(item['statistics'].get('viewCount', 0))
            likes = int(item['statistics'].get('likeCount', 0))
            comment_count = int(item['statistics'].get('commentCount', 0))
            published_at = item['snippet']['publishedAt']
            return title, views, likes, comment_count, published_at
        return None, None, None, None, None
    except Exception as e:
        print(f"Error al obtener detalles para el video {video_id}: {e}")
        return None, None, None, None, None

def get_batch_video_details(video_ids):
    """Obtiene detalles de múltiples videos (máximo 50) en una sola llamada a la API."""
    details_map = {}
    if not video_ids:
        return details_map
    try:
        # Unimos los IDs de los videos separados por comas
        ids_str = ",".join(video_ids)
        request = youtube.videos().list(part='snippet,statistics', id=ids_str)
        response = request.execute()
        if response and 'items' in response:
            for item in response['items']:
                vid = item['id']
                title = item['snippet']['title']
                views = int(item['statistics'].get('viewCount', 0))
                likes = int(item['statistics'].get('likeCount', 0))
                comment_count = int(item['statistics'].get('commentCount', 0))
                published_at = item['snippet']['publishedAt']
                details_map[vid] = (title, views, likes, comment_count, published_at)
    except Exception as e:
        print(f"Error en la consulta por lotes de la API: {e}")
    return details_map

def update_metrics():
    """Actualiza las métricas de todos los videos en la playlist utilizando consultas por lotes eficientes de 50 en 50."""
    global playlist_df
    if playlist_df.empty:
        print("La playlist está vacía. No hay métricas para actualizar.")
        return

    print(f"Actualizando vistas, likes, comentarios y métricas ganadas de {len(playlist_df)} canciones en lotes de 50...")
    import pytz
    new_york_timezone = pytz.timezone('America/New_York')
    now_ny = pd.Timestamp.now(tz=new_york_timezone)
    current_month_start_ny = now_ny.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
    end_of_previous_month_ny = current_month_start_ny - pd.Timedelta(microseconds=1)
    start_of_previous_seven_days_ny = current_month_start_ny - pd.Timedelta(days=7)

    is_reset_day = now_ny.day == 1

    # Lógica de ranking mensual en el día del reinicio
    rank_map_gained_for_reset = {}
    if is_reset_day:
        temp_df_for_rank = playlist_df.copy().dropna(subset=['views_gained'])
        if not temp_df_for_rank.empty:
            temp_df_for_rank = temp_df_for_rank.sort_values(by='views_gained', ascending=False).reset_index(drop=True)
            temp_df_for_rank['current_rank_gained'] = temp_df_for_rank.index + 1
            rank_map_gained_for_reset = temp_df_for_rank.set_index('video_id')['current_rank_gained'].to_dict()

    # Consultar la API de YouTube en lotes de hasta 50 IDs
    all_video_ids = playlist_df['video_id'].dropna().tolist()
    api_data_map = {}
    for i in range(0, len(all_video_ids), 50):
        batch_ids = all_video_ids[i:i+50]
        batch_data = get_batch_video_details(batch_ids)
        api_data_map.update(batch_data)

    # Procesar y mapear los datos de vuelta a cada fila del DataFrame
    for index, row in playlist_df.iterrows():
        video_id = row['video_id']
        if video_id not in api_data_map:
            # Si un video específico no se encontró en la API
            if pd.notna(video_id):
                print(f"Advertencia: No se pudo obtener información de la API para {video_id} (puede estar borrado o privado).")
            continue

        api_title, raw_api_views, raw_api_likes, raw_api_comment_count, published_at_str = api_data_map[video_id]
        published_at = pd.to_datetime(published_at_str)
        published_at_ny = published_at.tz_localize(new_york_timezone) if published_at.tz is None else published_at.tz_convert(new_york_timezone)

        # Un video se considera candidato a [NEW] por fecha de estreno
        is_new_video_by_date = (published_at_ny >= current_month_start_ny) or \
                               (published_at_ny >= start_of_previous_seven_days_ny and published_at_ny <= end_of_previous_month_ny)

        final_views = raw_api_views
        final_likes = raw_api_likes
        final_comment_count = raw_api_comment_count
        final_views_gained = 0
        final_likes_gained = 0
        final_comment_gained = 0

        # Caso 1: Es el día de reinicio mensual y aún no ha sido procesado este mes
        if is_reset_day and (pd.isna(row['last_reset_date']) or row['last_reset_date'].month != now_ny.month or row['last_reset_date'].year != now_ny.year):
            print(f"Reiniciando métricas para el video '{row['title']}' (ID: {video_id}) para el nuevo mes.")
            playlist_df.at[index, 'saved_views'] = raw_api_views
            playlist_df.at[index, 'saved_likes'] = raw_api_likes
            playlist_df.at[index, 'saved_comment_count'] = raw_api_comment_count

            playlist_df.at[index, 'views_gained'] = 0
            playlist_df.at[index, 'likes_gained'] = 0
            playlist_df.at[index, 'comment_gained'] = 0
            playlist_df.at[index, 'last_reset_date'] = now_ny.normalize()
            playlist_df.at[index, 'saved_rank_gained'] = rank_map_gained_for_reset.get(video_id, pd.NA)

            final_views_gained = 0
            final_likes_gained = 0
            final_comment_gained = 0

        # Caso 2: Es un video de estreno que todavía no ha pasado por el reinicio mensual
        elif is_new_video_by_date and (pd.isna(row['last_reset_date']) or row['last_reset_date'].month != now_ny.month or row['last_reset_date'].year != now_ny.year):
            playlist_df.at[index, 'saved_views'] = 0
            playlist_df.at[index, 'saved_likes'] = 0
            playlist_df.at[index, 'saved_comment_count'] = 0

            final_views_gained = raw_api_views
            final_likes_gained = raw_api_likes
            final_comment_gained = raw_api_comment_count

        # Caso 3: Actualización de cualquier otro video (antiguo o ya reiniciado)
        else:
            if not is_new_video_by_date and (row['saved_views'] == 0 or pd.isna(row['saved_views'])):
                playlist_df.at[index, 'saved_views'] = raw_api_views
                playlist_df.at[index, 'saved_likes'] = raw_api_likes
                playlist_df.at[index, 'saved_comment_count'] = raw_api_comment_count
                current_saved_views_for_gain_calc = raw_api_views
                current_saved_likes_for_gain_calc = raw_api_likes
                current_saved_comment_count_for_gain_calc = raw_api_comment_count
            else:
                current_saved_views_for_gain_calc = row['saved_views'] if pd.notna(row['saved_views']) else raw_api_views
                current_saved_likes_for_gain_calc = row['saved_likes'] if pd.notna(row['saved_likes']) else raw_api_likes
                current_saved_comment_count_for_gain_calc = row['saved_comment_count'] if pd.notna(row['saved_comment_count']) else raw_api_comment_count

                if pd.isna(row['saved_views']): playlist_df.at[index, 'saved_views'] = raw_api_views
                if pd.isna(row['saved_likes']): playlist_df.at[index, 'saved_likes'] = raw_api_likes
                if pd.isna(row['saved_comment_count']): playlist_df.at[index, 'saved_comment_count'] = raw_api_comment_count

            final_views_gained = raw_api_views - current_saved_views_for_gain_calc
            final_likes_gained = raw_api_likes - current_saved_likes_for_gain_calc
            final_comment_gained = raw_api_comment_count - current_saved_comment_count_for_gain_calc

        if (row['title'] == api_title) or (pd.isna(row['title']) and api_title is not None):
            playlist_df.at[index, 'title'] = api_title

        playlist_df.at[index, 'views'] = final_views
        playlist_df.at[index, 'likes'] = final_likes
        playlist_df.at[index, 'comment_count'] = final_comment_count
        playlist_df.at[index, 'published_at'] = published_at
        playlist_df.at[index, 'views_gained'] = final_views_gained
        playlist_df.at[index, 'likes_gained'] = final_likes_gained
        playlist_df.at[index, 'comment_gained'] = final_comment_gained

    print("Métricas actualizadas exitosamente.")
    save_playlist()

print(" [8] Cliente de la API de YouTube configurado y DataFrame de playlist inicializado/cargado de manera eficiente (Batching de 50 en 50). El código está completo.")

Playlist cargada desde /content/drive/MyDrive/Mi_Playlist_Vocaloid/playlist_data.csv
 [8] Cliente de la API de YouTube configurado y DataFrame de playlist inicializado/cargado de manera eficiente (Batching de 50 en 50). El código está completo.


In [38]:
# @title
from IPython.display import display, HTML
import pandas as pd
import pytz

try:
    from google.colab import data_table
except ImportError:
    pass

def display_top_125_gained():
    global playlist_df
    if playlist_df.empty:
        print("La playlist está vacía.")
        return

    filtered_df = playlist_df[playlist_df['views'].notna()].copy()
    if filtered_df.empty:
        print("No hay canciones válidas con vistas para procesar.")
        return

    top_125 = filtered_df.sort_values(by='views_gained', ascending=False).head(125).reset_index(drop=True)
    top_125['Posición'] = top_125.index + 1
    top_125['Cambio'] = 'N/A'

    new_york_timezone = pytz.timezone('America/New_York')
    now_ny = pd.Timestamp.now(tz=new_york_timezone)
    current_month_start_ny = now_ny.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
    end_of_previous_month_ny = current_month_start_ny - pd.Timedelta(microseconds=1)
    start_of_previous_seven_days_ny = current_month_start_ny - pd.Timedelta(days=7)

    for idx, row in top_125.iterrows():
        current_rank = row['Posición']
        previous_rank_gained = row['saved_rank_gained']
        last_reset = row['last_reset_date']

        has_been_reset_this_month = pd.notna(last_reset) and (last_reset.month == now_ny.month) and (last_reset.year == now_ny.year)
        is_new_by_date = False
        if pd.notna(row['published_at']):
            published_at = pd.to_datetime(row['published_at'])
            published_at_ny = published_at.tz_localize(new_york_timezone) if published_at.tz is None else published_at.tz_convert(new_york_timezone)
            if (published_at_ny >= current_month_start_ny) or \
               (published_at_ny >= start_of_previous_seven_days_ny and published_at_ny <= end_of_previous_month_ny):
                is_new_by_date = True

        if is_new_by_date and not has_been_reset_this_month:
            top_125.at[idx, 'Cambio'] = "[NEW]"
            continue

        if pd.notna(previous_rank_gained):
            change = int(previous_rank_gained) - int(current_rank)
            if change > 0:
                top_125.at[idx, 'Cambio'] = f"↑{change}"
            elif change < 0:
                top_125.at[idx, 'Cambio'] = f"↓{abs(change)}"
            else:
                top_125.at[idx, 'Cambio'] = "—"
        else:
            top_125.at[idx, 'Cambio'] = "—"

    def format_number_with_dots(num):
        if pd.isna(num):
            return "N/A"
        return f"{int(num):,}".replace(",", ".")

    def format_date(date_val):
        if pd.isna(date_val):
            return "N/A"
        return pd.to_datetime(date_val).strftime('%m-%d-%Y')

    def format_previous_rank(rank_val, has_been_reset_this_month, is_new_by_date):
        if is_new_by_date and not has_been_reset_this_month:
            return "[NEW]"
        if pd.isna(rank_val):
            return "—"
        return str(int(rank_val))

    display_df = top_125.copy()
    display_df['Vistas Actuales'] = display_df['views'].apply(format_number_with_dots)
    display_df['Vistas Ganadas'] = display_df['views_gained'].apply(format_number_with_dots)
    display_df['Me Gusta'] = display_df['likes'].apply(format_number_with_dots)
    display_df['Comentarios'] = display_df['comment_count'].apply(format_number_with_dots)
    display_df['Fecha de Publicación'] = display_df['published_at'].apply(format_date)
    display_df['Miniatura'] = display_df['video_id'].apply(lambda vid: f'<img src="https://img.youtube.com/vi/{vid}/default.jpg" width="75" height="56" style="border-radius:4px;" />' if pd.notna(vid) else '')

    pos_anterior_list = []
    for idx, row in top_125.iterrows():
        last_reset = row['last_reset_date']
        has_been_reset_this_month = pd.notna(last_reset) and (last_reset.month == now_ny.month) and (last_reset.year == now_ny.year)
        is_new_by_date = False
        if pd.notna(row['published_at']):
            published_at = pd.to_datetime(row['published_at'])
            published_at_ny = published_at.tz_localize(new_york_timezone) if published_at.tz is None else published_at.tz_convert(new_york_timezone)
            if (published_at_ny >= current_month_start_ny) or \
               (published_at_ny >= start_of_previous_seven_days_ny and published_at_ny <= end_of_previous_month_ny):
                is_new_by_date = True
        pos_anterior_list.append(format_previous_rank(row['saved_rank_gained'], has_been_reset_this_month, is_new_by_date))

    display_df['Posición Anterior'] = pos_anterior_list
    display_df = display_df.rename(columns={'title': 'Título', 'composer': 'Compositor', 'singer': 'Cantante'})
    display_df = display_df.reset_index(drop=True)
    display_df.index = display_df.index + 1
    display_df.index.name = 'Posición'

    print("\n--- TOP 125 Mensualizado por Vistas Ganadas (Consolidado) ---")
    ordered_columns = ['Cambio', 'Miniatura', 'Título', 'Compositor', 'Cantante', 'Fecha de Publicación', 'Vistas Actuales', 'Vistas Ganadas', 'Me Gusta', 'Comentarios', 'Posición Anterior']
    html_table = display_df[ordered_columns].to_html(escape=False, justify='center')
    display(HTML(html_table))

    top_100_subset = top_125.head(100).copy()

    def get_standardization_mapping(series_col):
        all_names = series_col.astype(str).str.split(',').explode().str.strip().replace('', pd.NA).dropna()
        if all_names.empty: return {}
        counts = all_names.value_counts()
        grouped_by_lower = {}
        for name, count in counts.items():
            name_lower = name.lower()
            if name_lower not in grouped_by_lower:
                grouped_by_lower[name_lower] = []
            grouped_by_lower[name_lower].append((name, count))
        standard_map = {}
        for name_lower, variants in grouped_by_lower.items():
            best_variant = sorted(variants, key=lambda x: x[1], reverse=True)[0][0]
            standard_map[name_lower] = best_variant
        return standard_map

    composer_std_map = get_standardization_mapping(top_100_subset['composer'])
    singer_std_map = get_standardization_mapping(top_100_subset['singer'])

    def standardize_list(cell_val, std_map):
        if pd.isna(cell_val): return cell_val
        parts = [p.strip() for p in str(cell_val).split(',') if p.strip()]
        standardized_parts = [std_map.get(p.lower(), p) for p in parts]
        return ", ".join(standardized_parts)

    top_100_subset['composer_std'] = top_100_subset['composer'].apply(lambda x: standardize_list(x, composer_std_map))
    top_100_subset['singer_std'] = top_100_subset['singer'].apply(lambda x: standardize_list(x, singer_std_map))

    print("\n--- New Releases Summary (Top 100) ---")
    new_songs_top_100 = top_100_subset[top_100_subset['Cambio'] == '[NEW]']
    if not new_songs_top_100.empty:
        for idx, row in new_songs_top_100.iterrows():
            producer_name = row['composer'] if pd.notna(row['composer']) else 'N/A'
            print(f"  - {row['title']} - {producer_name}")
    else:
        print("No hay nuevas canciones con la etiqueta [NEW] en el Top 100 actualmente.")

    print("\n--- Resumen de Compositores (Top 100 por Vistas Ganadas - Estandarizado) ---")
    if not top_100_subset['composer_std'].dropna().empty:
        processed_composers = top_100_subset['composer_std'].astype(str).str.split(',').explode().str.strip().replace('', pd.NA).dropna()
        composer_counts = processed_composers.value_counts()
        repeated_composers = composer_counts[composer_counts >= 3]
        if not repeated_composers.empty:
            for composer, count in repeated_composers.items():
                print(f"  - {composer} - {count} Appearances")
        else:
            print("Ningún compositor tiene 3 o más canciones en el Top 100 del subset.")

    print("\n--- Resumen de Cantantes (Top 100 por Vistas Ganadas - Estandarizado) ---")
    if not top_100_subset['singer_std'].dropna().empty:
        processed_singers = top_100_subset['singer_std'].astype(str).str.split(',').explode().str.strip().replace('', pd.NA).dropna()
        singer_counts = processed_singers.value_counts()
        repeated_singers = singer_counts[singer_counts >= 3]
        if not repeated_singers.empty:
            for singer, count in repeated_singers.items():
                print(f"  - {singer} - {count} Appearances")
        else:
            print("Ningún cantante tiene 3 o más canciones en el Top 100 del subset.")

    def parse_cambio_to_numeric(cambio_str):
        if not isinstance(cambio_str, str): return 0
        if cambio_str.startswith('↑'):
            try: return int(cambio_str[1:])
            except: return 0
        elif cambio_str.startswith('↓'):
            try: return -int(cambio_str[1:])
            except: return 0
        return 0

    top_100_analytics = top_100_subset.copy()
    top_100_analytics['cambio_num'] = top_100_analytics['Cambio'].apply(parse_cambio_to_numeric)
    changed_songs = top_100_analytics[top_100_analytics['cambio_num'] != 0].copy()

    print("\n--- Top 7 Canciones con Mayor Ganancia de Puestos (Top 100) ---")
    if not changed_songs.empty:
        gains = changed_songs.sort_values(by='cambio_num', ascending=False).head(7)
        if not gains.empty:
            for idx, row in gains.iterrows():
                print(f"  - {row['title']} - {row['Cambio']} spots")
        else:
            print("No hay canciones con ganancia de puestos registrada para mostrar.")
    else:
        print("No hay suficientes datos de cambios de posición aún.")

    print("\n--- Top 7 Canciones con Mayor Caída de Puestos (Top 100) ---")
    if not changed_songs.empty:
        losses = changed_songs.sort_values(by='cambio_num', ascending=True).head(7)
        if not losses.empty:
            for idx, row in losses.iterrows():
                print(f"  - {row['title']} - {row['Cambio']} spots")
        else:
            print("No hay canciones con caída de puestos registrada para mostrar.")
    else:
        print("No hay suficientes datos de cambios de posición aún.")

    display_time_until_reset()

print("[9] VISUALIZACIÓN EXCLUSIVA: TOP 125 MENSUAL Y RESÚMENES ARTÍSTICOS ESTANDARIZADOS")

[9] VISUALIZACIÓN EXCLUSIVA: TOP 125 MENSUAL Y RESÚMENES ARTÍSTICOS ESTANDARIZADOS


In [39]:
# @title
# Añade o actualiza tus videos aquí. Ejecuta esta celda cada vez que quieras modificar tu playlist.

add_video_to_playlist('https://www.youtube.com/watch?v=dI1j38zsx98', titulo_personalizado='Mystery', composer='MIMI', singer='Kasane Teto')

add_video_to_playlist('https://www.youtube.com/watch?v=8Fy73FE2ik0', titulo_personalizado='Slumber Party', composer='miyu', singer='Kasane Teto, Otomachi Una')
add_video_to_playlist('https://www.youtube.com/watch?v=vnzuVECDD_E', titulo_personalizado='Seito yume nippo', composer='kinoue64', singer='Hatsune Miku')
add_video_to_playlist('https://www.youtube.com/watch?v=-S_WxH8RmkA', titulo_personalizado='Taker {Decrease}', composer='Haraguchi Sasuke', singer='Kasane Teto')
add_video_to_playlist('https://www.youtube.com/watch?v=O_jX3LFSy30', titulo_personalizado='So So Afraid', composer='MIMI', singer='Kasane Teto')
add_video_to_playlist('https://www.youtube.com/watch?v=-jH80xvHX-E', titulo_personalizado='Emerald (SEKAI ver.)', composer='Tatsuya Kitani', singer='Hatsune Miku')
add_video_to_playlist('https://www.youtube.com/watch?v=g5mQ6VIKm34', titulo_personalizado='Override (Daibaku Hashin REMIX)', composer='Daibaku Hashin REMIX', singer='Kasane Teto')
add_video_to_playlist('https://www.youtube.com/watch?v=0YHJoexs5TU', titulo_personalizado='Mirror Mirror', composer='TAK', singer='Hatsune Miku, Kasane Teto')
add_video_to_playlist('https://www.youtube.com/watch?v=wQNs1l0U7rI', titulo_personalizado='Deliver that special, knockout blow!', composer='Sakaki Ness', singer='Hatsune Miku')
add_video_to_playlist('https://www.youtube.com/watch?v=OlprURqnZw8', titulo_personalizado='My Lovely Nook', composer='ABM', singer='Otomachi Una')
add_video_to_playlist('https://www.youtube.com/watch?v=z9FZ7sLzLj4', titulo_personalizado='#StanningAMagicalGirl', composer='nogi', singer='Chifuyu Hanakuma')
add_video_to_playlist('https://www.youtube.com/watch?v=Bi2gm5crERs', titulo_personalizado='Boozehound', composer='Satsuki', singer='Kasane Teto, Yi Xi')
add_video_to_playlist('https://www.youtube.com/watch?v=8oiyutRI3nI', titulo_personalizado='Playback', composer='somethings off with my vocaloid', singer='Hatsune Miku, Ui')
add_video_to_playlist('https://www.youtube.com/watch?v=931lT5zGBy8', titulo_personalizado='Searching for Stars ♭♭', composer='Haraguchi Sasuke', singer='Megurine Luka')


Video 'Mystery' añadido.
Video 'Slumber Party' añadido.
Video 'Seito yume nippo' añadido.


/tmp/ipykernel_8361/3553975497.py:111: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  playlist_df = pd.concat([playlist_df, new_row], ignore_index=True)
/tmp/ipykernel_8361/3553975497.py:111: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  playlist_df = pd.concat([playlist_df, new_row], ignore_index=True)
/tmp/ipykernel_8361/3553975497.py:111: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or 

Video 'Taker {Decrease}' añadido.
Video 'So So Afraid' añadido.
Video 'Emerald (SEKAI ver.)' añadido.
Video 'Override (Daibaku Hashin REMIX)' añadido.
Video 'Mirror Mirror' añadido.
Video 'Deliver that special, knockout blow!' añadido.


/tmp/ipykernel_8361/3553975497.py:111: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  playlist_df = pd.concat([playlist_df, new_row], ignore_index=True)
/tmp/ipykernel_8361/3553975497.py:111: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  playlist_df = pd.concat([playlist_df, new_row], ignore_index=True)
/tmp/ipykernel_8361/3553975497.py:111: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or 

Video 'My Lovely Nook' añadido.
Video '#StanningAMagicalGirl' añadido.
Video 'Boozehound' añadido.
Video 'Playback' añadido.
Video 'Searching for Stars ♭♭' añadido.


/tmp/ipykernel_8361/3553975497.py:111: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  playlist_df = pd.concat([playlist_df, new_row], ignore_index=True)
/tmp/ipykernel_8361/3553975497.py:111: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  playlist_df = pd.concat([playlist_df, new_row], ignore_index=True)


In [40]:
# @title
print("\n--- Actualizando Métricas y mostrando la tabla de vistas totales (N/A filtrados) ---")

update_metrics()
display_playlist()
save_playlist()


--- Actualizando Métricas y mostrando la tabla de vistas totales (N/A filtrados) ---
Actualizando vistas, likes, comentarios y métricas ganadas de 882 canciones en lotes de 50...
Métricas actualizadas exitosamente.
Playlist guardada en /content/drive/MyDrive/Mi_Playlist_Vocaloid/playlist_data.csv

--- Tu Playlist (ordenada por vistas) ---


,Cambio,Título,Compositor,Cantante,Fecha de Publicación,Vistas Actuales,Vistas Ganadas,Me Gusta,Me Gusta Ganados,Comentarios,Comentarios Ganados
Posición,,,,,,,,,,,
1,—,Mesmerizer,Satsuki,"Hatsune Miku, Kasane Teto",04-27-2024,231.419.935,1.547.780,2.257.740,7.530,133.640,262
2,—,Young Girl A,siinamota,Kagamine Rin,10-17-2013,230.604.316,1.701.736,3.057.064,12.676,223.710,203
3,—,Goodbye Sengen,Chinozo,Flower,04-13-2020,161.401.718,289.653,1.425.163,1.498,87.465,82
4,—,Monitoring,DECO*27,Hatsune Miku,11-22-2024,138.954.420,1.031.105,1.350.628,4.616,22.721,62
5,—,Tetoris,Hiiragi Magnetite,Kasane Teto,11-08-2024,138.486.625,949.105,1.026.880,3.523,24.918,67
...,...,...,...,...,...,...,...,...,...,...,...
878,[NEW],#StanningAMagicalGirl,nogi,Chifuyu Hanakuma,11-06-2026,136.313,136.313,8.289,8.289,285,285
879,↓13,FAKE UP,ePiaeon,Hatsune Miku,04-11-2025,133.199,2.043,5.275,20,323,2
880,[NEW],Boozehound,Satsuki,"Kasane Teto, Yi Xi",11-06-2026,128.378,128.378,7.528,7.528,345,345


Playlist guardada en /content/drive/MyDrive/Mi_Playlist_Vocaloid/playlist_data.csv


In [41]:
# @title
print("\n--- Actualizando Métricas y mostrando la tabla de vistas ganadas (N/A filtrados) ---")

display_playlist_by_views_gained() # Muestra la playlist ordenada por vistas ganadas


--- Actualizando Métricas y mostrando la tabla de vistas ganadas (N/A filtrados) ---

--- Tu Playlist Completa (Vistas Ganadas - Ranking Mensualizado Interactivo) ---


,Cambio,Título,Compositor,Cantante,Fecha de Publicación,Vistas (Crecimiento),Me Gusta,Comentarios,Posición Anterior
Posición,,,,,,,,,
1,—,YARARARA,ABM,Kasane Teto,05-25-2026,+3.143.301 (72.755.222),934.270,26.669,1
2,[NEW],Playback,somethings off with my vocaloid,"Hatsune Miku, Ui",11-06-2026,+1.907.307 (1.907.307),67.637,5.060,[NEW]
3,↑1,Young Girl A,siinamota,Kagamine Rin,10-17-2013,+1.701.736 (230.604.316),3.057.064,223.710,4
4,↓1,Looping the Rooms,rusino,Hatsune Miku,12-18-2025,+1.675.774 (82.579.148),1.156.048,20.610,3
5,↓3,Displayholic,ABM,Hatsune Miku,09-28-2026,+1.602.954 (9.464.780),167.082,10.972,2
...,...,...,...,...,...,...,...,...,...
878,↓16,UNICYCLE,Pechiko,"Hatsune Miku, Kasane Teto",07-14-2026,+987 (367.300),14.254,0,862
879,↓15,Hystera Hysteria,PPP Sounds (LOL),KAFU,12-08-2023,+956 (170.179),6.596,194,864
880,↓14,SAW - BRAINSTORM,SAWTOWNE,Hatsune Miku,07-07-2023,+669 (110.348),3.087,94,866


In [42]:
# @title
print("\n--- Actualizando Métricas y mostrando la tabla de vistas ganadas (N/A filtrados) ---")
display_top_125_gained()


--- Actualizando Métricas y mostrando la tabla de vistas ganadas (N/A filtrados) ---

--- TOP 125 Mensualizado por Vistas Ganadas (Consolidado) ---


,Cambio,Miniatura,Título,Compositor,Cantante,Fecha de Publicación,Vistas Actuales,Vistas Ganadas,Me Gusta,Comentarios,Posición Anterior
Posición,,,,,,,,,,,
1,—,,YARARARA,ABM,Kasane Teto,05-25-2026,72.755.222,3.143.301,934.270,26.669,1
2,[NEW],,Playback,somethings off with my vocaloid,"Hatsune Miku, Ui",11-06-2026,1.907.307,1.907.307,67.637,5.060,[NEW]
3,↑1,,Young Girl A,siinamota,Kagamine Rin,10-17-2013,230.604.316,1.701.736,3.057.064,223.710,4
4,↓1,,Looping the Rooms,rusino,Hatsune Miku,12-18-2025,82.579.148,1.675.774,1.156.048,20.610,3
5,↓3,,Displayholic,ABM,Hatsune Miku,09-28-2026,9.464.780,1.602.954,167.082,10.972,2
6,↓1,,Mesmerizer,Satsuki,"Hatsune Miku, Kasane Teto",04-27-2024,231.419.935,1.547.780,2.257.740,133.640,5
7,[NEW],,My Lovely Nook,ABM,Otomachi Una,11-06-2026,1.536.426,1.536.426,65.118,5.085,[NEW]
8,↓2,,Brain Rot,Tokyo Manaka,Kasane Teto,02-25-2026,50.258.641,1.491.728,762.344,14.910,6
9,↓1,,Retry Now,NAKISO,Hatsune Miku,08-01-2025,87.535.899,1.243.924,1.014.869,28.003,8



--- New Releases Summary (Top 100) ---
  - Playback - somethings off with my vocaloid
  - My Lovely Nook - ABM
  - Emerald (SEKAI ver.) - Tatsuya Kitani
  - Mirror Mirror - TAK
  - Mystery - MIMI
  - Deliver that special, knockout blow! - Sakaki Ness
  - Searching for Stars ♭♭ - Haraguchi Sasuke
  - Taker {Decrease} - Haraguchi Sasuke
  - Slumber Party - miyu
  - Seito yume nippo - kinoue64
  - So So Afraid - MIMI

--- Resumen de Compositores (Top 100 por Vistas Ganadas - Estandarizado) ---
  - DECO*27 - 6 Appearances
  - FLAVOR FOLEY - 5 Appearances
  - ABM - 4 Appearances
  - PinocchioP - 4 Appearances
  - Haraguchi Sasuke - 4 Appearances
  - Jamie Paige - 3 Appearances
  - HALLO CEL - 3 Appearances
  - TAK - 3 Appearances

--- Resumen de Cantantes (Top 100 por Vistas Ganadas - Estandarizado) ---
  - Hatsune Miku - 51 Appearances
  - Kasane Teto - 39 Appearances
  - Kagamine Rin - 5 Appearances
  - GUMI - 4 Appearances
  - Otomachi Una - 3 Appearances
  - Ui - 3 Appearances
  - Kaai